In [ ]:
!pip install torch torchvision torchaudio
!pip install matplotlib
!pip install numpy
!pip install pillow
!pip install opencv-python
!pip install scipy
!pip install imageio
!pip install tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils
from torchvision.datasets import CelebA
from PIL import Image
import os
import numpy as np
import matplotlib.pyplot as plt
import torchvision.utils as vutils
import cv2
import imageio
from tqdm import tqdm
import scipy

In [ ]:
from __future__ import print_function

import argparse
import os
import random
import torch
import torch.nn.parallel
import torch.backends.cudnn as cudnn
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML


manualSeed = 999

print("Random Seed: ", manualSeed)
random.seed(manualSeed)
torch.manual_seed(manualSeed)

In [ ]:
class RandomLabelDataset(Dataset):
    def __init__(self, image_folder, transform=None, num_classes=10):
        self.image_folder = image_folder
        self.transform = transform
        self.image_paths = [os.path.join(image_folder, img) for img in os.listdir(image_folder)]
        self.num_classes = num_classes

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        # Assign a random label between 1 and num_classes (inclusive)
        random_label = random.randint(1, self.num_classes)
        
        return image, random_label  # Image and Random Label

# Image transformations
transform = transforms.Compose([
    transforms.Resize(128),
    transforms.CenterCrop(128),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

# Dataset and DataLoader
dataset = RandomLabelDataset(image_folder="/kaggle/input/cardetection/car/train/images", transform=transform)
dataloader = DataLoader(dataset, batch_size=1024, shuffle=True)

In [ ]:
for batch in dataloader:
    images, random_labels = batch  # Only unpack image and label
    print("Batch Loaded!")
    print("Images Shape:", images.shape)  # Should be [batch_size, channels, height, width]
    print("Random Labels:", random_labels[:10])  # Print first 10 labels to check if it's working properly
    break  # Exit after first batch test


In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
device

In [ ]:
workers = 4
batch_size = 512
nc = 3  # Number of channels (RGB)
nz = 100  # Size of latent vector
ngf = 64  # Size of feature maps in generator
ndf = 64  # Size of feature maps in discriminator
num_epochs = 5
lr = 0.0002
beta1 = 0.5
ngpu = 1  # Number of GPUs
device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

In [ ]:
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.2)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.2)
        nn.init.constant_(m.bias.data, 0)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F # Need this for softmax
import numpy as np
import time # For progress indication
from collections.abc import Iterable # To check for iterable basis

# --- Keep your Generator class exactly as it was ---
class Generator(nn.Module):
    def __init__(self, ngpu, nz, ngf, nc, epsilon=0.1):
        super(Generator, self).__init__()
        self.ngpu = ngpu
        self.epsilon = epsilon

        self.main = nn.Sequential(
            nn.ConvTranspose2d(nz, ngf * 16, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 16), nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 16, ngf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 8), nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4), nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2), nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf), nn.ReLU(True),
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def jacobian_noise(self, input_tensor):
        input_clone = input_tensor.clone().detach().requires_grad_(True)
        with torch.enable_grad():
            output = self.main(input_clone)
            loss = output.norm()
            if input_clone.grad is not None: input_clone.grad.zero_()
            loss.backward()
        if input_clone.grad is None: return input_tensor.detach()
        gradients = input_clone.grad.data
        perturbed_input = input_tensor.detach() + self.epsilon * gradients.sign()
        perturbed_input = torch.clamp(perturbed_input, -1, 1)
        return perturbed_input

    def forward(self, input_tensor):
        if torch.is_grad_enabled():
            perturbed_input = self.jacobian_noise(input_tensor)
            output = self.main(perturbed_input)
        else:
            output = self.main(input_tensor)
        return output


In [ ]:

class Discriminator(nn.Module):
    def __init__(self, ngpu, nc, ndf, num_classes=10):
        super(Discriminator, self).__init__()
        self.ngpu = ngpu
        self.num_classes = num_classes # Store num_classes
        self.main = nn.Sequential(
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 8, ndf * 16, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 16), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 16, ndf * 32, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ndf * 32), nn.LeakyReLU(0.2, inplace=True)
        )
        self.adv_layer = nn.Conv2d(ndf * 32, 1, 1, 1, 0, bias=False)
        self.classifier = nn.Conv2d(ndf * 32, num_classes, 1, 1, 0, bias=False)

    def forward(self, input):
        features = self.main(input)
        realness_score = self.adv_layer(features).view(-1, 1)
        class_scores = self.classifier(features).view(-1, self.num_classes)
        return realness_score, class_scores

    # --- ADDED: SImBA Attack Method ---
    def _get_class_probability(self, x, true_label):
        """ Helper to get softmax probability of the true label. """
        with torch.no_grad():
            _, class_scores = self.forward(x) # Get logits from classifier head
            probabilities = F.softmax(class_scores, dim=1)
            prob = probabilities[:, true_label].item() # Get probability for the true class
        return prob

    def _generate_basis_vectors(self, num_dims, num_vectors, device):
        """ Generates random basis vectors (normalized). """
        basis = torch.randn(num_vectors, num_dims, device=device)
        basis_norm = torch.norm(basis, p=2, dim=1, keepdim=True) + 1e-8
        basis = basis / basis_norm
        return basis

    def apply_simba_attack(self, image_tensor, true_label, device,
                           step_size=0.1, max_iterations=1000,
                           basis_type='random', # Can be 'random' or 'pixels'
                           pixel_stride=16): # Used only if basis_type='pixels'
        """
        Applies SImBA attack targeting this Discriminator's classifier head.

        Args:
            image_tensor (torch.Tensor): Single input image tensor [1, C, H, W], normalized [-1, 1].
            true_label (int): The true integer class label for the image.
            device (torch.device): Device for calculations.
            step_size (float): Epsilon value for probing steps.
            max_iterations (int): Maximum number of queries/iterations.
            basis_type (str): 'random' for random directions, 'pixels' for standard basis.
            pixel_stride (int): How many pixels to skip when using 'pixels' basis.

        Returns:
            torch.Tensor: The perturbed adversarial image tensor.
        """
        self.eval() # Ensure model is in evaluation mode

        x_orig = image_tensor.clone().detach().to(device)
        x_adv = image_tensor.clone().detach().to(device)
        num_dims = x_adv.numel() # Total number of features (pixels * channels)
        batch_size, C, H, W = x_adv.shape

        # Get initial probability
        current_prob = self._get_class_probability(x_adv, true_label)

        print(f"Starting SImBA attack... Initial P(true_label={true_label}): {current_prob:.4f}")
        start_time = time.time()
        num_queries = 0

        # --- Prepare Basis ---
        if basis_type == 'random':
            # Generate a pool of random directions (can reuse or regenerate)
            num_basis_vectors = min(max_iterations, num_dims) # Generate enough for budget
            basis_vectors = self._generate_basis_vectors(num_dims, num_basis_vectors, device)
            basis_vectors = basis_vectors.view(num_basis_vectors, C, H, W) # Reshape
            indices = torch.randperm(num_basis_vectors, device=device) # Shuffle order
        elif basis_type == 'pixels':
            # Create indices for pixel basis (potentially strided)
            indices = [(c, h, w) for c in range(C)
                       for h in range(0, H, pixel_stride)
                       for w in range(0, W, pixel_stride)]
            np.random.shuffle(indices) # Shuffle order
            num_basis_vectors = len(indices)
        else:
            raise ValueError("basis_type must be 'random' or 'pixels'")

        # --- Iterative Probing ---
        for i in range(min(max_iterations, num_basis_vectors)):
            if num_queries >= max_iterations:
                print("Query budget reached.")
                break

            # Select basis vector 'q'
            if basis_type == 'random':
                q = basis_vectors[indices[i]].unsqueeze(0) # Add batch dim
            else: # basis_type == 'pixels'
                c, h, w = indices[i]
                q = torch.zeros_like(x_adv)
                q[0, c, h, w] = 1.0 # Standard basis vector

            # Probe in positive direction
            x_plus = torch.clamp(x_adv + step_size * q, -1.0, 1.0)
            prob_plus = self._get_class_probability(x_plus, true_label)
            num_queries += 1

            if prob_plus < current_prob: # Improvement found
                x_adv = x_plus
                current_prob = prob_plus
                # print(f"Iter {i+1}/{num_basis_vectors}, Query {num_queries}: + Prob -> {current_prob:.4f}") # Verbose
                continue # Move to next basis vector

            if num_queries >= max_iterations:
                print("Query budget reached.")
                break

            # Probe in negative direction (only if positive didn't improve)
            x_minus = torch.clamp(x_adv - step_size * q, -1.0, 1.0)
            prob_minus = self._get_class_probability(x_minus, true_label)
            num_queries += 1

            if prob_minus < current_prob: # Improvement found
                x_adv = x_minus
                current_prob = prob_minus
                # print(f"Iter {i+1}/{num_basis_vectors}, Query {num_queries}: - Prob -> {current_prob:.4f}") # Verbose

            # Optional: Check for misclassification and early stop
            # with torch.no_grad():
            #     _, current_scores = self.forward(x_adv)
            #     current_pred = torch.argmax(current_scores, dim=1).item()
            # if current_pred != true_label:
            #     print(f"Misclassification achieved at iteration {i+1}, query {num_queries}.")
            #     break

        total_time = time.time() - start_time
        print(f"SImBA attack finished in {total_time:.2f}s ({num_queries} queries). Final P(true_label={true_label}): {current_prob:.4f}")
        return x_adv.detach()


In [ ]:
# This is how it should be to properly utilize GPU when available:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
print(f"Using device: {device}")
print(f"Is CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Current CUDA device: {torch.cuda.current_device()}")
    print(f"Device name: {torch.cuda.get_device_name()}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import numpy as np
import time
import gc
import os # Added for creating samples directory



# Parameters
nz = 100  # Size of z latent vector
ngf = 64  # Size of feature maps in generator
ndf = 64  # Size of feature maps in discriminator
nc = 3    # Number of channels (RGB)
ngpu = 1  # Number of GPUs (set to 0 if no GPU)
num_classes = 10 # Number of classes in your dataset
device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")
print(f"Using device: {device}")

# Create the models
netG = Generator(ngpu, nz, ngf, nc, epsilon=0.1).to(device) # epsilon for jacobian_noise
netD = Discriminator(ngpu, nc, ndf, num_classes=num_classes).to(device)

# Apply weights_init
netG.apply(weights_init)
netD.apply(weights_init)

# Handle multi-GPU if desired
if (device.type == 'cuda') and (ngpu > 1):
    netG = nn.DataParallel(netG, list(range(ngpu)))
    netD = nn.DataParallel(netD, list(range(ngpu)))

# Loss functions and optimizers
adversarial_loss = nn.BCEWithLogitsLoss() # More stable than BCELoss + Sigmoid
classification_loss = nn.CrossEntropyLoss()
optimizerD = optim.Adam(netD.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=0.0002, betas=(0.5, 0.999))

# Fixed noise for visualization
fixed_noise = torch.randn(64, nz, 1, 1, device=device)

# Training parameters
num_epochs = 5 # Set number of epochs
display_step = 10 # Print status every N batches
running_loss_D = 0.0 # For tracking average loss per epoch
running_loss_G = 0.0 # For tracking average loss per epoch

# Create samples directory if it doesn't exist
samples_dir = "samples"
os.makedirs(samples_dir, exist_ok=True)

print("Starting Training Loop...")
start_time = time.time()
iters = 0
current_epoch = 0 # Initialize epoch counter

try:
    # Main training loop
    for epoch in range(current_epoch, num_epochs):
        current_epoch = epoch # Store current epoch for checkpointing
        epoch_start_time = time.time()
        for i, batch in enumerate(dataloader):
            # Get batch data
            real_images = batch[0].to(device)

            # Ensure labels are provided and are Long type for CrossEntropyLoss
            if len(batch) > 1:
                # Ensure labels are within [0, num_classes-1] and correct type
                real_labels = (batch[1] % num_classes).long().to(device)
            else:
                # Generate random labels if not provided by dataloader
                real_labels = torch.randint(0, num_classes, (real_images.size(0),), dtype=torch.long, device=device)

            batch_size = real_images.size(0)

            # Create labels for adversarial loss
            real_label_adv = torch.full((batch_size, 1), 1.0, dtype=torch.float, device=device)
            fake_label_adv = torch.full((batch_size, 1), 0.0, dtype=torch.float, device=device)

            #######################
            # Train Discriminator #
            #######################
            netD.zero_grad()

            # --- Train with all-real batch ---
            real_pred_adv, real_pred_class = netD(real_images)
            errD_real_adv = adversarial_loss(real_pred_adv, real_label_adv)
            errD_real_class = classification_loss(real_pred_class, real_labels)
            errD_real = errD_real_adv + errD_real_class
            # Calculate gradients for Discriminator in backward pass
            # We calculate gradients separately for real and fake batches
            # before calling optimizerD.step() to avoid potential issues
            # with graph retention if needed, although here it's straightforward.
            errD_real.backward()
            D_x_adv = real_pred_adv.mean().item() # Avg D output for real images
            D_x_class_acc = (real_pred_class.argmax(1) == real_labels).float().mean().item() # Accuracy on real images

            # --- Train with all-fake batch ---
            # Generate fake image batch with G
            noise = torch.randn(batch_size, nz, 1, 1, device=device)
            # Detach fake images so gradients don't flow back to G during D training
            fake_images = netG(noise).detach()
            fake_pred_adv, _ = netD(fake_images) # Ignore classification output for fake images in D loss
            errD_fake_adv = adversarial_loss(fake_pred_adv, fake_label_adv)
            # Calculate gradients for this batch
            errD_fake_adv.backward()
            D_G_z1 = fake_pred_adv.mean().item() # Avg D output for fake images (before D update)
            # Add the gradients from the all-real and all-fake batches
            errD = errD_real + errD_fake_adv # Total Discriminator loss

            # Update Discriminator
            optimizerD.step()


            ###################
            # Train Generator #
            ###################
            netG.zero_grad()

            # Generate a NEW batch of fake images (gradients WILL flow back to G now)
            noise = torch.randn(batch_size, nz, 1, 1, device=device)
            fake_images = netG(noise) # Forward pass through G (will use jacobian_noise)
            fake_pred_adv, fake_pred_class = netD(fake_images) # Forward pass fake data through D

            # Calculate G's loss based on this output
            # 1. Adversarial Loss: G wants D to think the fakes are real
            errG_adv = adversarial_loss(fake_pred_adv, real_label_adv) # Use real labels (1.0)

            # 2. Classification Loss: G wants D to classify the fakes correctly
            #    Generate target labels for the fake images (can be random or specific)
            target_labels = torch.randint(0, num_classes, (batch_size,), dtype=torch.long, device=device)
            errG_class = classification_loss(fake_pred_class, target_labels)

            # Combine losses for G
            errG = errG_adv + errG_class

            # Calculate gradients for G and Update G
            errG.backward()
            D_G_z2 = fake_pred_adv.mean().item() # Avg D output for fake images (after D update, before G update)
            optimizerG.step()

            # Update running losses for epoch average calculation
            running_loss_D += errD.item()
            running_loss_G += errG.item()

            # --- Output training stats ---
            if i % display_step == 0:
                elapsed = time.time() - start_time
                print(f'[{epoch}/{num_epochs}][{i}/{len(dataloader)}] | '
                      f'Time: {elapsed:.1f}s | Loss_D: {errD.item():.4f} | Loss_G: {errG.item():.4f} | '
                      f'D(x): {D_x_adv:.4f} | D(G(z)): {D_G_z1:.4f} / {D_G_z2:.4f} | '
                      f'D Acc Real: {D_x_class_acc*100:.2f}%')

            # --- Checkpointing and Visualization ---
            if (iters % 500 == 0) or ((epoch == num_epochs-1) and (i == len(dataloader)-1)):
                print(f"Saving samples and checkpoint at iteration {iters}...")
                # --- Generate and Save Fake Images ---
                with torch.no_grad(): # IMPORTANT: Use no_grad for visualization
                    # This call to netG will now correctly use the 'else' path in forward()
                    fake_viz = netG(fixed_noise).detach().cpu()
                img_grid = vutils.make_grid(fake_viz, padding=2, normalize=True)

                plt.figure(figsize=(8, 8))
                plt.axis("off")
                plt.title(f"Fake Images - Epoch {epoch}, Iter {iters}")
                plt.imshow(np.transpose(img_grid, (1, 2, 0)))
                plt.savefig(os.path.join(samples_dir, f"fake_images_epoch_{epoch}_iter_{iters}.png"))
                plt.close()

                # --- Save Checkpoint ---
                torch.save({
                    'netG': netG.state_dict(),
                    'netD': netD.state_dict(),
                    'optimizerG': optimizerG.state_dict(),
                    'optimizerD': optimizerD.state_dict(),
                    'epoch': epoch,
                    'iters': iters,
                    'fixed_noise': fixed_noise,
                }, f"checkpoint_iter_{iters}.pth")

            iters += 1

            # Optional: Free memory periodically (might help on resource-constrained systems)
            if i % 100 == 0:
                gc.collect()
                if device.type == 'cuda':
                    torch.cuda.empty_cache()

        # --- End of Epoch ---
        epoch_time = time.time() - epoch_start_time
        avg_loss_D = running_loss_D / len(dataloader)
        avg_loss_G = running_loss_G / len(dataloader)
        print(f"===> Epoch {epoch} completed in {epoch_time:.2f}s | Avg Loss_D: {avg_loss_D:.4f} | Avg Loss_G: {avg_loss_G:.4f}")
        running_loss_D = 0.0 # Reset for next epoch
        running_loss_G = 0.0 # Reset for next epoch

    print("Training finished!")

    # --- Save final models ---
    print("Saving final models...")
    torch.save(netG.state_dict(), "generator_final.pth")
    torch.save(netD.state_dict(), "discriminator_final.pth")

except Exception as e:
    print(f"\n!!! Training interrupted by error: {e} !!!\n")
    # Save checkpoint in case of error
    print("Saving checkpoint at crash...")
    try:
        torch.save({
            'netG': netG.state_dict(),
            'netD': netD.state_dict(),
            'optimizerG': optimizerG.state_dict(),
            'optimizerD': optimizerD.state_dict(),
            'epoch': current_epoch, # Save the epoch it crashed in
            'iters': iters,
            'fixed_noise': fixed_noise,
        }, "checkpoint_at_crash.pth")
        print("Checkpoint saved to checkpoint_at_crash.pth")
    except Exception as save_e:
        print(f"!!! Failed to save checkpoint: {save_e} !!!")
    import traceback # Import traceback for detailed error info
    traceback.print_exc() # Print the full traceback
print("Training finished!")


# Save the final models
torch.save(netG.state_dict(), 'generator.pth')
torch.save(netD.state_dict(), 'discriminator.pth')

In [ ]:
# class SimpleClassifier(nn.Module):
#     def __init__(self, nc=3, num_classes=10):
#         super(SimpleClassifier, self).__init__()
#         self.conv1 = nn.Conv2d(nc, 16, kernel_size=3, padding=1)
#         self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
#         self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
#         self.fc1 = nn.Linear(64 * 8 * 8, 128)
#         self.fc2 = nn.Linear(128, num_classes)

#     def forward(self, x):
#         x = F.relu(self.conv1(x))
#         x = F.max_pool2d(x, 2)
#         x = F.relu(self.conv2(x))
#         x = F.max_pool2d(x, 2)
#         x = F.relu(self.conv3(x))
#         x = F.max_pool2d(x, 2)
#         x = x.view(x.size(0), -1)
#         x = F.relu(self.fc1(x))
#         x = self.fc2(x)
#         return x

In [ ]:
# def jacobian_saliency_map_attack(model, input_tensor, target_class, epsilon=0.1):
#     """
#     Applies JSMA to generate adversarial noise using the Jacobian-based saliency map.
#     """
#     input_tensor.requires_grad = True
#     output = model(input_tensor)
    
#     # Compute the Jacobian matrix
#     model.zero_grad()
#     target_score = output[:, target_class]
#     target_score.backward()
#     jacobian = input_tensor.grad  # Gradient with respect to input
    
#     # Compute saliency map
#     saliency_map = jacobian.abs()
#     max_saliency, _ = torch.max(saliency_map.view(saliency_map.size(0), -1), dim=1)
#     perturbation = epsilon * (saliency_map / (max_saliency.view(-1, 1, 1, 1) + 1e-8))
    
#     # Apply perturbation
#     adversarial_example = torch.clamp(input_tensor + perturbation, 0, 1)
#     return adversarial_example.detach()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import time
import os

def visualize_msimba_attack(model_path, image_path, true_label, device, 
                           step_size=0.1, max_iterations=1000,
                           basis_type='random', pixel_stride=16,
                           num_steps_to_save=5, display_freq=100):
    """
    Visualizes the MSImBA (Multi-Step SImBA) attack on an image using the classifier
    head of a Discriminator model.
    
    Args:
        model_path (str): Path to saved Discriminator model state_dict
        image_path (str): Path to the input image
        true_label (int): True class label for the image
        device (torch.device): Device to use for computations
        step_size (float): Epsilon value for perturbation steps
        max_iterations (int): Maximum number of iterations/queries
        basis_type (str): 'random' for random directions, 'pixels' for standard basis
        pixel_stride (int): Stride for pixel basis (used with basis_type='pixels')
        num_steps_to_save (int): Number of intermediate steps to save for animation
        display_freq (int): How often to display progress
        
    Returns:
        dict: Contains original image, perturbed images at different steps, and attack metrics
    """
    if not os.path.exists(model_path):
        print(f"Error: Discriminator model file not found at {model_path}")
        return None
    if not os.path.exists(image_path):
        print(f"Error: Image file not found at {image_path}")
        return None
        
    # --- Load the model ---
    print(f"Loading Discriminator model from {model_path}...")
    # Define model parameters (should match your trained model)
    ndf = 64
    nc = 3
    ngpu = 1
    num_classes = 10  # Adjust based on your model
    
    # Instantiate the discriminator
    model = Discriminator(ngpu, nc, ndf, num_classes).to(device)
    
    # Load the state dictionary
    checkpoint = torch.load(model_path, map_location=device)
    
    # Handle potential DataParallel wrapping
    if isinstance(checkpoint, dict) and 'netD' in checkpoint:
        # This is a training checkpoint that includes the discriminator
        state_dict = checkpoint['netD']
    else:
        # This is just the discriminator state_dict
        state_dict = checkpoint
        
    # Remove 'module.' prefix if present (from DataParallel)
    if list(state_dict.keys())[0].startswith('module.'):
        from collections import OrderedDict
        new_state_dict = OrderedDict()
        for k, v in state_dict.items():
            name = k[7:]  # remove 'module.'
            new_state_dict[name] = v
        model.load_state_dict(new_state_dict)
    else:
        model.load_state_dict(state_dict)
        
    model.eval()  # Set to evaluation mode
    
    # --- Load and preprocess the image ---
    print(f"Loading and preprocessing image: {image_path}...")
    # Image transformation (adjust according to your training setup)
    transform = transforms.Compose([
        transforms.Resize((128, 128)),  # Adjust size to match your model's input
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # Normalize to [-1, 1]
    ])
    
    try:
        image = Image.open(image_path).convert('RGB')
        image_tensor = transform(image).unsqueeze(0).to(device)  # Add batch dim
    except Exception as e:
        print(f"Error loading or transforming image: {e}")
        return None
        
    # --- Apply MSImBA attack ---
    print(f"Starting MSImBA attack (max {max_iterations} iterations)...")
    
    # Make a copy of the original image for attack
    x_adv = image_tensor.clone().detach()
    
    # Store results for visualization
    results = {
        'original_image': tensor_to_np(image_tensor),
        'perturbed_images': [],
        'probabilities': [],
        'predictions': [],
        'queries': [],
        'iterations': []
    }
    
    # Calculate intervals for saving images
    save_interval = max(1, max_iterations // (num_steps_to_save - 1))
    
    # Get initial prediction and probability
    with torch.no_grad():
        _, class_scores = model(x_adv)
        probs = F.softmax(class_scores, dim=1)
        initial_prob = probs[0, true_label].item()
        initial_pred = torch.argmax(class_scores, dim=1).item()
        
    print(f"Initial prediction: {initial_pred}, P(true_label={true_label}): {initial_prob:.4f}")
    
    # Add initial state to results
    results['perturbed_images'].append(tensor_to_np(x_adv))
    results['probabilities'].append(initial_prob)
    results['predictions'].append(initial_pred)
    results['queries'].append(0)
    results['iterations'].append(0)
    
    # Apply the attack
    try:
        x_adv = model.apply_simba_attack(
            image_tensor, true_label, device,
            step_size=step_size, max_iterations=max_iterations,
            basis_type=basis_type, pixel_stride=pixel_stride
        )
    except Exception as e:
        print(f"Error during attack: {e}")
        import traceback
        traceback.print_exc()
        return results
        
    # Get final prediction and probability
    with torch.no_grad():
        _, class_scores = model(x_adv)
        probs = F.softmax(class_scores, dim=1)
        final_prob = probs[0, true_label].item()
        final_pred = torch.argmax(class_scores, dim=1).item()
        
    print(f"Final prediction: {final_pred}, P(true_label={true_label}): {final_prob:.4f}")
    
    # Add final state to results
    results['perturbed_images'].append(tensor_to_np(x_adv))
    results['probabilities'].append(final_prob)
    results['predictions'].append(final_pred)
    results['queries'].append(max_iterations)
    results['iterations'].append(max_iterations)
    
    print("Attack visualization data collection complete.")
    return results



In [ ]:
def tensor_to_np(tensor):
    """Convert a tensor to numpy array for visualization"""
    img = tensor.squeeze(0).detach().cpu()  # Remove batch dim, move to CPU
    img = img * 0.5 + 0.5  # De-normalize from [-1, 1] to [0, 1]
    img = torch.clamp(img, 0, 1)  # Ensure range is valid
    img_np = img.numpy()
    # Transpose from (C, H, W) to (H, W, C) for Matplotlib
    return np.transpose(img_np, (1, 2, 0))



In [ ]:
def plot_msimba_results(results, save_path="msimba_visualization.png"):
    """
    Plot and save the visualization of MSImBA attack results
    
    Args:
        results (dict): The results from visualize_msimba_attack
        save_path (str): Path to save the visualization image
    """
    if results is None or len(results['perturbed_images']) < 2:
        print("No valid results to visualize")
        return
        
    # Create a figure with original image, final perturbed image, and metrics
    plt.figure(figsize=(15, 8))
    
    # Original image
    plt.subplot(2, 3, 1)
    plt.imshow(results['original_image'])
    plt.title("Original Image")
    plt.axis('off')
    
    # Final perturbed image
    plt.subplot(2, 3, 2)
    plt.imshow(results['perturbed_images'][-1])
    plt.title(f"Perturbed Image\n(after {results['iterations'][-1]} iterations)")
    plt.axis('off')
    
    # Difference (perturbation)
    perturbation = results['perturbed_images'][-1] - results['original_image']
    # Normalize the difference for better visualization
    perturbation = (perturbation - perturbation.min()) / (perturbation.max() - perturbation.min() + 1e-8)
    
    plt.subplot(2, 3, 3)
    plt.imshow(perturbation)
    plt.title("Perturbation\n(normalized for visibility)")
    plt.axis('off')
    
    # Plot probability curve
    plt.subplot(2, 3, 4)
    plt.plot(results['iterations'], results['probabilities'], marker='o')
    plt.title(f"Probability of True Class ({results['predictions'][0]})")
    plt.xlabel("Iterations")
    plt.ylabel("Probability")
    plt.grid(True)
    
    # Plot class prediction changes if any
    plt.subplot(2, 3, 5)
    plt.plot(results['iterations'], results['predictions'], marker='o')
    plt.title("Predicted Class")
    plt.xlabel("Iterations")
    plt.ylabel("Class")
    plt.grid(True)
    
    # Add metrics text
    plt.subplot(2, 3, 6)
    plt.axis('off')
    
    info_text = (
        f"Attack Summary:\n\n"
        f"Initial prediction: {results['predictions'][0]}\n"
        f"Final prediction: {results['predictions'][-1]}\n\n"
        f"Initial probability: {results['probabilities'][0]:.4f}\n"
        f"Final probability: {results['probabilities'][-1]:.4f}\n\n"
        f"Probability reduction: {results['probabilities'][0] - results['probabilities'][-1]:.4f}\n"
        f"Total iterations: {results['iterations'][-1]}\n"
    )
    
    plt.text(0.1, 0.5, info_text, fontsize=12, va='center')
    
    plt.tight_layout()
    plt.savefig(save_path)
    print(f"Visualization saved to {save_path}")
    plt.show()



In [ ]:
def run_msimba_visualization(discriminator_path, image_path, true_label, 
                            max_iterations=500, step_size=0.1, 
                            basis_type='random', save_path="msimba_results.png"):
    """
    Run the MSImBA attack visualization pipeline
    
    Args:
        discriminator_path (str): Path to the saved discriminator model
        image_path (str): Path to the image to attack
        true_label (int): True class label of the image
        max_iterations (int): Maximum iterations for the attack
        step_size (float): Step size for the attack
        basis_type (str): 'random' or 'pixels'
        save_path (str): Path to save the visualization
    """
    # Determine device
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # Apply MSImBA attack and collect visualization data
    results = visualize_msimba_attack(
        discriminator_path, image_path, true_label, device,
        step_size=step_size, max_iterations=max_iterations,
        basis_type=basis_type
    )
    
    # Plot and save results
    if results:
        plot_msimba_results(results, save_path)
    else:
        print("Failed to generate MSImBA visualization")

In [ ]:
# Usage example (uncomment and modify paths/parameters as needed):

discriminator_path = "/kaggle/working/discriminator_final.pth"
image_path = "/kaggle/input/cardetection/car/test/images/00000_00000_00021_png.rf.0803f8eb5b4566c44100cfef9d0bfa8e.jpg"
true_label = 3  # The true class index for your image
max_iterations = 100 # Number of iterations for the attack
step_size = 0.1  # Perturbation magnitude
basis_type = 'random'  # 'random' or 'pixels'

run_msimba_visualization(
    discriminator_path, image_path, true_label, 
    max_iterations=max_iterations, step_size=step_size,
    basis_type=basis_type, save_path="msimba_visualization.png"
)


In [ ]:
def modified_apply_simba_attack(self, image_tensor, true_label, device,
                             step_size=0.1, max_iterations=1000,
                             basis_type='random', pixel_stride=16, 
                             collect_intermediate=True, intermediate_freq=50):
    """
    An enhanced version of the SImBA attack that collects intermediate results
    for visualization.
    
    Args:
        image_tensor (torch.Tensor): Single input image tensor [1, C, H, W], normalized to [-1, 1]
        true_label (int): The true class label for the image
        device (torch.device): Device for calculations
        step_size (float): Epsilon value for probing steps
        max_iterations (int): Maximum number of iterations/queries
        basis_type (str): 'random' for random directions, 'pixels' for standard basis
        pixel_stride (int): Stride for pixel basis (used with basis_type='pixels')
        collect_intermediate (bool): Whether to collect intermediate results
        intermediate_freq (int): How often to save intermediate results
        
    Returns:
        dict: Contains perturbed image tensor and intermediate results
    """
    self.eval()  # Ensure model is in evaluation mode
    
    x_orig = image_tensor.clone().detach().to(device)
    x_adv = image_tensor.clone().detach().to(device)
    num_dims = x_adv.numel()  # Total number of features (pixels * channels)
    batch_size, C, H, W = x_adv.shape
    
    # For storing intermediate results
    intermediate_results = {
        'iterations': [],
        'queries': [],
        'perturbed_images': [],
        'probabilities': [],
        'predictions': []
    }
    
    # Get initial probability
    current_prob = self._get_class_probability(x_adv, true_label)
    current_query = 0
    
    # Record initial state
    if collect_intermediate:
        intermediate_results['iterations'].append(0)
        intermediate_results['queries'].append(0)
        intermediate_results['perturbed_images'].append(x_adv.clone().detach())
        intermediate_results['probabilities'].append(current_prob)
        with torch.no_grad():
            _, current_scores = self.forward(x_adv)
            current_pred = torch.argmax(current_scores, dim=1).item()
        intermediate_results['predictions'].append(current_pred)
    
    print(f"Starting MSImBA attack... Initial P(true_label={true_label}): {current_prob:.4f}")
    start_time = time.time()
    
    # --- Prepare Basis ---
    if basis_type == 'random':
        # Generate random basis vectors
        num_basis_vectors = min(max_iterations, num_dims)
        basis_vectors = self._generate_basis_vectors(num_dims, num_basis_vectors, device)
        basis_vectors = basis_vectors.view(num_basis_vectors, C, H, W)
        indices = torch.randperm(num_basis_vectors, device=device)
    elif basis_type == 'pixels':
        # Create pixel basis indices
        indices = [(c, h, w) for c in range(C) 
                  for h in range(0, H, pixel_stride) 
                  for w in range(0, W, pixel_stride)]
        np.random.shuffle(indices)
        num_basis_vectors = len(indices)
    else:
        raise ValueError("basis_type must be 'random' or 'pixels'")
    
    # --- Iterative Probing ---
    for i in range(min(max_iterations, num_basis_vectors)):
        if current_query >= max_iterations:
            print("Query budget reached.")
            break
        
        # Select basis vector
        if basis_type == 'random':
            q = basis_vectors[indices[i]].unsqueeze(0)
        else:  # basis_type == 'pixels'
            c, h, w = indices[i]
            q = torch.zeros_like(x_adv)
            q[0, c, h, w] = 1.0
        
        # Probe in positive direction
        x_plus = torch.clamp(x_adv + step_size * q, -1.0, 1.0)
        prob_plus = self._get_class_probability(x_plus, true_label)
        current_query += 1
        
        if prob_plus < current_prob:  # Improvement found
            x_adv = x_plus
            current_prob = prob_plus
            
            # Record intermediate state if specified
            if collect_intermediate and i % intermediate_freq == 0:
                intermediate_results['iterations'].append(i + 1)
                intermediate_results['queries'].append(current_query)
                intermediate_results['perturbed_images'].append(x_adv.clone().detach())
                intermediate_results['probabilities'].append(current_prob)
                with torch.no_grad():
                    _, current_scores = self.forward(x_adv)
                    current_pred = torch.argmax(current_scores, dim=1).item()
                intermediate_results['predictions'].append(current_pred)
            
            continue  # Move to next basis vector
        
        if current_query >= max_iterations:
            print("Query budget reached.")
            break
        
        # Probe in negative direction
        x_minus = torch.clamp(x_adv - step_size * q, -1.0, 1.0)
        prob_minus = self._get_class_probability(x_minus, true_label)
        current_query += 1
        
        if prob_minus < current_prob:  # Improvement found
            x_adv = x_minus
            current_prob = prob_minus
            
            # Record intermediate state if specified
            if collect_intermediate and i % intermediate_freq == 0:
                intermediate_results['iterations'].append(i + 1)
                intermediate_results['queries'].append(current_query)
                intermediate_results['perturbed_images'].append(x_adv.clone().detach())
                intermediate_results['probabilities'].append(current_prob)
                with torch.no_grad():
                    _, current_scores = self.forward(x_adv)
                    current_pred = torch.argmax(current_scores, dim=1).item()
                intermediate_results['predictions'].append(current_pred)
        
        # Check for misclassification every 50 iterations
        if i % 50 == 0:
            with torch.no_grad():
                _, current_scores = self.forward(x_adv)
                current_pred = torch.argmax(current_scores, dim=1).item()
                if current_pred != true_label:
                    print(f"Misclassification achieved at iteration {i+1}, query {current_query}.")
    
    # Record final state if not already recorded
    if collect_intermediate:
        if intermediate_results['iterations'][-1] != i + 1:
            intermediate_results['iterations'].append(i + 1)
            intermediate_results['queries'].append(current_query)
            intermediate_results['perturbed_images'].append(x_adv.clone().detach())
            intermediate_results['probabilities'].append(current_prob)
            with torch.no_grad():
                _, current_scores = self.forward(x_adv)
                current_pred = torch.argmax(current_scores, dim=1).item()
            intermediate_results['predictions'].append(current_pred)
    
    total_time = time.time() - start_time
    print(f"MSImBA attack finished in {total_time:.2f}s ({current_query} queries).")
    print(f"Final P(true_label={true_label}): {current_prob:.4f}")
    
    return x_adv.detach(), intermediate_results


In [ ]:
def visualize_msimba_attack(model, image_tensor, true_label, device,
                           step_size=0.1, max_iterations=1000,
                           basis_type='random', pixel_stride=16, 
                           collect_freq=50):
    """
    An enhanced version of the SImBA attack that collects intermediate results
    for visualization.
    
    Args:
        model (nn.Module): Your trained Discriminator model
        image_tensor (torch.Tensor): Single input image tensor [1, C, H, W], normalized to [-1, 1]
        true_label (int): The true class label for the image
        device (torch.device): Device for calculations
        step_size (float): Epsilon value for probing steps
        max_iterations (int): Maximum number of iterations/queries
        basis_type (str): 'random' for random directions, 'pixels' for standard basis
        pixel_stride (int): Stride for pixel basis (used with basis_type='pixels')
        collect_freq (int): How often to save intermediate results
        
    Returns:
        tuple: (perturbed image tensor, visualization results dictionary)
    """
    model.eval()  # Ensure model is in evaluation mode
    
    x_orig = image_tensor.clone().detach().to(device)
    x_adv = image_tensor.clone().detach().to(device)
    num_dims = x_adv.numel()  # Total number of features (pixels * channels)
    batch_size, C, H, W = x_adv.shape
    
    # For storing visualization results
    viz_results = {
        'iterations': [],
        'queries': [],
        'perturbed_images': [],
        'probabilities': [],
        'predictions': []
    }
    
    # Get initial probability
    with torch.no_grad():
        _, class_scores = model(x_adv)
        probs = F.softmax(class_scores, dim=1)
        current_prob = probs[0, true_label].item()
        current_pred = torch.argmax(class_scores, dim=1).item()
    
    current_query = 0
    
    # Record initial state
    viz_results['iterations'].append(0)
    viz_results['queries'].append(0)
    viz_results['perturbed_images'].append(x_adv.clone().detach())
    viz_results['probabilities'].append(current_prob)
    viz_results['predictions'].append(current_pred)
    
    print(f"Starting MSImBA attack... Initial P(true_label={true_label}): {current_prob:.4f}")
    start_time = time.time()
    
    # --- Prepare Basis ---
    if basis_type == 'random':
        # Generate random basis vectors
        num_basis_vectors = min(max_iterations, num_dims)
        basis = torch.randn(num_basis_vectors, num_dims, device=device)
        basis_norm = torch.norm(basis, p=2, dim=1, keepdim=True) + 1e-8
        basis = basis / basis_norm
        basis_vectors = basis.view(num_basis_vectors, C, H, W)
        indices = torch.randperm(num_basis_vectors, device=device)
    elif basis_type == 'pixels':
        # Create pixel basis indices
        indices = [(c, h, w) for c in range(C) 
                  for h in range(0, H, pixel_stride) 
                  for w in range(0, W, pixel_stride)]
        np.random.shuffle(indices)
        num_basis_vectors = len(indices)
    else:
        raise ValueError("basis_type must be 'random' or 'pixels'")
    
    # --- Iterative Probing ---
    for i in range(min(max_iterations, num_basis_vectors)):
        if current_query >= max_iterations:
            print("Query budget reached.")
            break
        
        # Select basis vector
        if basis_type == 'random':
            q = basis_vectors[indices[i]].unsqueeze(0)
        else:  # basis_type == 'pixels'
            c, h, w = indices[i]
            q = torch.zeros_like(x_adv)
            q[0, c, h, w] = 1.0
        
        # Probe in positive direction
        x_plus = torch.clamp(x_adv + step_size * q, -1.0, 1.0)
        
        with torch.no_grad():
            _, class_scores = model(x_plus)
            probs = F.softmax(class_scores, dim=1)
            prob_plus = probs[0, true_label].item()
        
        current_query += 1
        
        if prob_plus < current_prob:  # Improvement found
            x_adv = x_plus
            current_prob = prob_plus
            
            # Record intermediate state if it's time
            if i % collect_freq == 0:
                with torch.no_grad():
                    _, current_scores = model(x_adv)
                    current_pred = torch.argmax(current_scores, dim=1).item()
                
                viz_results['iterations'].append(i + 1)
                viz_results['queries'].append(current_query)
                viz_results['perturbed_images'].append(x_adv.clone().detach())
                viz_results['probabilities'].append(current_prob)
                viz_results['predictions'].append(current_pred)
            
            continue  # Move to next basis vector
        
        if current_query >= max_iterations:
            print("Query budget reached.")
            break
        
        # Probe in negative direction
        x_minus = torch.clamp(x_adv - step_size * q, -1.0, 1.0)
        
        with torch.no_grad():
            _, class_scores = model(x_minus)
            probs = F.softmax(class_scores, dim=1)
            prob_minus = probs[0, true_label].item()
        
        current_query += 1
        
        if prob_minus < current_prob:  # Improvement found
            x_adv = x_minus
            current_prob = prob_minus
            
            # Record intermediate state if it's time
            if i % collect_freq == 0:
                with torch.no_grad():
                    _, current_scores = model(x_adv)
                    current_pred = torch.argmax(current_scores, dim=1).item()
                
                viz_results['iterations'].append(i + 1)
                viz_results['queries'].append(current_query)
                viz_results['perturbed_images'].append(x_adv.clone().detach())
                viz_results['probabilities'].append(current_prob)
                viz_results['predictions'].append(current_pred)
        
        # Check for misclassification and record data at regular intervals
        if i % collect_freq == 0:
            with torch.no_grad():
                _, current_scores = model(x_adv)
                current_pred = torch.argmax(current_scores, dim=1).item()
                
                # Only record if we haven't just recorded this iteration
                if len(viz_results['iterations']) == 0 or viz_results['iterations'][-1] != i + 1:
                    viz_results['iterations'].append(i + 1)
                    viz_results['queries'].append(current_query)
                    viz_results['perturbed_images'].append(x_adv.clone().detach())
                    viz_results['probabilities'].append(current_prob)
                    viz_results['predictions'].append(current_pred)
                    
                if current_pred != true_label:
                    print(f"Misclassification achieved at iteration {i+1}, query {current_query}.")
    
    # Record final state if not already recorded
    if viz_results['iterations'][-1] != i + 1:
        with torch.no_grad():
            _, current_scores = model(x_adv)
            current_pred = torch.argmax(current_scores, dim=1).item()
        
        viz_results['iterations'].append(i + 1)
        viz_results['queries'].append(current_query)
        viz_results['perturbed_images'].append(x_adv.clone().detach())
        viz_results['probabilities'].append(current_prob)
        viz_results['predictions'].append(current_pred)
    
    total_time = time.time() - start_time
    print(f"MSImBA attack finished in {total_time:.2f}s ({current_query} queries).")
    print(f"Final P(true_label={true_label}): {current_prob:.4f}")
    
    return x_adv.detach(), viz_results


def plot_msimba_results(original_img, viz_results, save_path="msimba_visualization.png"):
    """
    Plot and save the visualization of MSImBA attack results
    
    Args:
        original_img (torch.Tensor): Original image tensor
        viz_results (dict): Results from visualize_msimba_attack
        save_path (str): Path to save the visualization
    """
    # Convert tensor to numpy if needed
    if isinstance(original_img, torch.Tensor):
        original_img = tensor_to_np(original_img)
    
    # Define a function to convert tensors in the results to numpy
    def process_images(images):
        return [tensor_to_np(img) if isinstance(img, torch.Tensor) else img for img in images]
    
    perturbed_images = process_images(viz_results['perturbed_images'])
    
    # Create a figure with multiple subplots
    plt.figure(figsize=(15, 10))
    
    # Plot original image
    plt.subplot(2, 3, 1)
    plt.imshow(original_img)
    plt.title("Original Image")
    plt.axis('off')
    
    # Plot final perturbed image
    plt.subplot(2, 3, 2)
    plt.imshow(perturbed_images[-1])
    plt.title(f"Perturbed Image\nAfter {viz_results['iterations'][-1]} iterations")
    plt.axis('off')
    
    # Plot perturbation (difference)
    perturbation = perturbed_images[-1] - original_img
    # Normalize for better visibility
    perturbation = (perturbation - perturbation.min()) / (perturbation.max() - perturbation.min() + 1e-8)
    
    plt.subplot(2, 3, 3)
    plt.imshow(perturbation)
    plt.title("Perturbation\n(normalized for visibility)")
    plt.axis('off')
    
    # Plot probability curve
    plt.subplot(2, 3, 4)
    plt.plot(viz_results['iterations'], viz_results['probabilities'], 'b-o')
    plt.title("True Class Probability")
    plt.xlabel("Iterations")
    plt.ylabel("Probability")
    plt.grid(True)
    
    # Plot prediction curve
    plt.subplot(2, 3, 5)
    plt.plot(viz_results['iterations'], viz_results['predictions'], 'r-o')
    plt.title("Predicted Class")
    plt.xlabel("Iterations")
    plt.ylabel("Class ID")
    plt.grid(True)
    
    # Text summary
    plt.subplot(2, 3, 6)
    plt.axis('off')
    info_text = (
        f"Attack Summary:\n\n"
        f"Initial prediction: {viz_results['predictions'][0]}\n"
        f"Final prediction: {viz_results['predictions'][-1]}\n\n"
        f"Initial probability: {viz_results['probabilities'][0]:.4f}\n"
        f"Final probability: {viz_results['probabilities'][-1]:.4f}\n\n"
        f"Probability reduction: {viz_results['probabilities'][0] - viz_results['probabilities'][-1]:.4f}\n"
        f"Total iterations: {viz_results['iterations'][-1]}\n"
        f"Total queries: {viz_results['queries'][-1]}"
    )
    plt.text(0.1, 0.5, info_text, fontsize=12, va='center')
    
    plt.tight_layout()
    plt.savefig(save_path)
    print(f"Visualization saved to {save_path}")
    plt.show()


def tensor_to_np(tensor):
    """Convert tensor to numpy for visualization"""
    img = tensor.squeeze(0).detach().cpu()  # Remove batch dim, move to CPU
    img = img * 0.5 + 0.5  # De-normalize from [-1, 1] to [0, 1]
    img = torch.clamp(img, 0, 1)  # Ensure range is valid
    img_np = img.numpy()
    # Transpose from (C, H, W) to (H, W, C) for Matplotlib
    return np.transpose(img_np, (1, 2, 0))


# Example usage:
import torch
import torchvision.transforms as transforms
from PIL import Image

# Load your trained discriminator
discriminator_path = "/kaggle/working/discriminator_final.pth"
checkpoint = torch.load(discriminator_path, map_location=device)
netD.load_state_dict(checkpoint)
netD.eval()

# Prepare an image to attack
image_path = "/kaggle/input/cardetection/car/test/images/00000_00000_00021_png.rf.0803f8eb5b4566c44100cfef9d0bfa8e.jpg"
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
image = Image.open(image_path).convert('RGB')
image_tensor = transform(image).unsqueeze(0).to(device)

# Set the true label for the image
true_label = 3  # Replace with actual label

# Run MSImBA attack with visualization
perturbed_image, viz_results = visualize_msimba_attack(
    netD, image_tensor, true_label, device,
    step_size=0.1, max_iterations=500,
    basis_type='random', collect_freq=50
)

# Plot and save visualization
plot_msimba_results(image_tensor, viz_results, "msimba_visualization.png")


In [ ]:
import os
import torch
from PIL import Image
import torchvision.transforms as transforms
import torchvision.utils as vutils
from tqdm import tqdm

# --- 1. Setup Directories & Datasets ---
# Define the source folders for both train and test
dataset_paths = {
    "train": "/kaggle/input/cardetection/car/train/images",
    "test":  "/kaggle/input/cardetection/car/test/images"
}

# Base directory where the subfolders will be created
base_output_dir = "/kaggle/working/adversarial_images"

# --- 2. Define Image Transformations ---
# Use the same transformations as your training process
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Ensure model is in eval mode
netD.eval()

# --- 3. Main Loop: Process both Train and Test ---
for split_name, input_dir in dataset_paths.items():
    
    # Create specific output directory (e.g., .../adversarial_images/train)
    current_output_dir = os.path.join(base_output_dir, split_name)
    os.makedirs(current_output_dir, exist_ok=True)

    print(f"\n🚀 Processing '{split_name}' dataset...")
    print(f"📂 Input: {input_dir}")
    print(f"💾 Output: {current_output_dir}")

    try:
        # Get list of images
        image_files = [f for f in os.listdir(input_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
        print(f"Found {len(image_files)} images in {split_name}.")
    except FileNotFoundError:
        print(f"⚠️ Error: The directory '{input_dir}' was not found. Skipping...")
        continue

    if not image_files:
        print(f"No images found in {split_name}.")
        continue

    # Process each image with a progress bar
    for image_name in tqdm(image_files, desc=f"Attacking {split_name}"):
        image_path = os.path.join(input_dir, image_name)
        # Save with original name or prefix it
        output_path = os.path.join(current_output_dir, f"adv_{image_name}")

        try:
            # Load and prepare the image
            image = Image.open(image_path).convert('RGB')
            image_tensor = transform(image).unsqueeze(0).to(device)

            # Determine the true label (using model prediction on clean image)
            with torch.no_grad():
                _, class_scores = netD(image_tensor)
                true_label = torch.argmax(class_scores, dim=1).item()

            # Apply SImBA Attack
            perturbed_tensor = netD.apply_simba_attack(
                image_tensor,
                true_label,
                device,
                step_size=0.2,      # Adjustable perturbation strength
                max_iterations=500, # Lower this if it takes too long per image
                basis_type='random'
            )

            # Save the perturbed image
            vutils.save_image(perturbed_tensor, output_path, normalize=True)

        except Exception as e:
            # If one image fails, print the error but keep going
            print(f"❌ Could not process {image_name}: {e}")

print("\n✅ Finished generating all adversarial images for Train and Test.")